<a href="https://colab.research.google.com/github/nmwaura4/Projects/blob/main/Text_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re
import nltk
import textblob
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from wordcloud import WordCloud, STOPWORDS, ImageColorGenerator
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize, sent_tokenize



In [ ]:
df=pd.read_excel('/content/Text_Data.xlsx')
df.head()

In [ ]:
df.describe().T

In [ ]:
def clean_text(text):
  text=str(text).lower()
  text=re.sub(r'\[.*?\]','',text)
  text=re.sub(r'https?://\S+|www\.\S+','',text)
df['clean']=df['Unnamed: 0'].apply(clean_text)

In [ ]:
nltk.download('punkt')
nltk.download('stopwords')

In [ ]:
# First, ensure the 'clean' column is correctly populated by fixing the clean_text logic
# and re-applying it. The original clean_text function in cell Mm_-SuyAwyz8 did not return a value.
def clean_text_corrected(text):
  text=str(text).lower()
  text=re.sub(r'\[.*?\]','',text)
  text=re.sub(r'https?://\S+|www\.\S+','',text)
  return text # Added return statement

df['clean'] = df['Unnamed: 0'].apply(clean_text_corrected)

# Now, proceed with stopword removal as originally intended
stopwords=nltk.corpus.stopwords.words('english')
df['clean']=df['clean'].apply(lambda x:' '.join([word for word in x.split() if word not in (stopwords)]))

In [ ]:
from textblob import TextBlob

df['polarity'] = df['clean'].apply(lambda x: TextBlob(x).sentiment.polarity)

df['sentimen'] = df['polarity'].apply(lambda x: 'positive' if x > 0 else ('negative' if x < 0 else 'neutral'))
df.head()

In [ ]:
df['sentimen'].value_counts().plot(kind='bar')
plt.title('Sentiment Label Distribution')

In [ ]:
from textblob import TextBlob
print(TextBlob("This is a positive statement.").sentiment)

In [ ]:
positive_sentiments = df[df['sentimen'] == 'positive']
display(positive_sentiments[['Unnamed: 0', 'sentimen']])

In [ ]:
neutral_sentiments = df[df['sentimen'] == 'neutral']
display(neutral_sentiments[['Unnamed: 0', 'sentimen']])

### Creating a Custom Lexicon: Negative Keywords

To improve sentiment analysis accuracy, especially for domain-specific language, creating a custom lexicon is crucial. This involves defining lists of words that carry specific positive or negative sentiment within your context. Below is an initial list of negative keywords based on common product/service issues.

In [ ]:
negative_keywords = [
    "leaking", "mortality", "low count", "cancel", "efficacy", "no label",
    "thrips", "persistent", "damaged", "poor", "bad", "unhappy", "failure",
    "broken", "defect", "problem", "issue", "complaint", "difficult",
    "ineffective", "unacceptable", "disappointing", "frustrated", "delay",
    "expensive", "waste", "error", "wrong", "weak", "struggle", "harmful",
    "unresponsive", "incomplete", "missing", "incorrect", "out of stock",
    "unreliable", "faulty", "non-functional", "unusable", "poor quality",
    "not working", "unpleasant", "deteriorated", "substandard", "inferior",
    "unmet", "discrepancy", "unfit", "compromised", "restricted", "slow"
]

print(f"Defined {len(negative_keywords)} negative keywords.")

In [ ]:
negative_keywords = [
    "leaking", "mortality", "low count", "cancel", "efficacy", "no label",
    "thrips", "persistent", "damaged", "poor", "bad", "unhappy", "failure",
    "broken", "defect", "problem", "issue", "complaint", "difficult",
    "ineffective", "unacceptable", "disappointing", "frustrated", "delay",
    "expensive", "waste", "error", "wrong", "weak", "struggle", "harmful",
    "unresponsive", "incomplete", "missing", "incorrect", "out of stock",
    "unreliable", "faulty", "non-functional", "unusable", "poor quality",
    "not working", "unpleasant", "deteriorated", "substandard", "inferior",
    "unmet", "discrepancy", "unfit", "compromised", "restricted", "slow"
]

# Function to check for negative keywords
def check_negative_keywords(text, keywords):
    for keyword in keywords:
        if keyword in text:
            return True
    return False

# Apply the function to create a new column indicating presence of negative keywords
df['has_negative_keyword'] = df['clean'].apply(lambda x: check_negative_keywords(x, negative_keywords))

# Update the sentiment based on custom lexicon
def update_sentiment_with_lexicon(row):
    if row['has_negative_keyword']:
        return 'negative'
    else:
        return row['sentimen'] # Keep TextBlob's sentiment if no negative keyword found

df['custom_sentiment'] = df.apply(update_sentiment_with_lexicon, axis=1)

display(df[['Unnamed: 0', 'sentimen', 'has_negative_keyword', 'custom_sentiment']].head())

In [ ]:
# Visualize the distribution of the new custom sentiment labels
df['custom_sentiment'].value_counts().plot(kind='bar', title='Custom Sentiment Label Distribution')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.show()

In [ ]:
print(df['custom_sentiment'].value_counts())

In [ ]:
negative_custom_sentiments = df[df['custom_sentiment'] == 'negative']
display(negative_custom_sentiments[['Unnamed: 0', 'custom_sentiment']])

### Categorizing Negative Sentiments by Keyword Frequency

To understand the primary drivers of negative sentiment, we'll analyze the frequency of the predefined negative keywords within the `negative_custom_sentiments` data. This will help us identify the most common issues reported.

In [ ]:
from collections import Counter

# Ensure negative_keywords is defined (it should be from previous cells, but good practice to include)
negative_keywords = [
    "leaking", "mortality", "low count", "cancel", "efficacy", "no label",
    "thrips", "persistent", "damaged", "poor", "bad", "unhappy", "failure",
    "broken", "defect", "problem", "issue", "complaint", "difficult",
    "ineffective", "unacceptable", "disappointing", "frustrated", "delay",
    "expensive", "waste", "error", "wrong", "weak", "struggle", "harmful",
    "unresponsive", "incomplete", "missing", "incorrect", "out of stock",
    "unreliable", "faulty", "non-functional", "unusable", "poor quality",
    "not working", "unpleasant", "deteriorated", "substandard", "inferior",
    "unmet", "discrepancy", "unfit", "compromised", "restricted", "slow"
]

# Initialize a counter for negative keywords
keyword_counts = Counter()

# Iterate through the 'clean' text of negative sentiments and count keyword occurrences
for text in negative_custom_sentiments['clean']:
    for keyword in negative_keywords:
        if keyword in text:
            keyword_counts[keyword] += 1

# Convert to DataFrame for easier plotting
keyword_df = pd.DataFrame(keyword_counts.items(), columns=['Keyword', 'Count']).sort_values(by='Count', ascending=False)

display(keyword_df.head(10)) # Display top 10 negative keywords

In [ ]:
# Visualize the distribution of top negative keywords
plt.figure(figsize=(10, 6))
sns.barplot(x='Count', y='Keyword', data=keyword_df.head(10), palette='viridis')
plt.title('Top 10 Most Frequent Negative Keywords in Negative Sentiments')
plt.xlabel('Frequency')
plt.ylabel('Negative Keyword')
plt.show()